## Import Data

In [1]:
import pandas as pd

#Order detailed
path = r"D:\Study\Amazon\Baby Products (meta)\Baby_Products.jsonl"
baby_product = pd.read_json(path, lines=True)  
baby_product.head()


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,4,Good buy for preschool naps and home use...,I bought two of these for my kids for nap time...,[],B004FM7VOW,B089MS68G8,AGKASBHYZPGTEPO6LWZPVJWB2BVA,2016-08-18 18:52:17,1,True
1,5,THEY WORK- and are super cute to boot...,LOVE THESE! AND THEY WORK!!! I was on the fenc...,[],B01E5E703G,B01E5E703G,AGKASBHYZPGTEPO6LWZPVJWB2BVA,2016-08-18 17:44:04,1,True
2,1,cute but small and pretty much unusable as a c...,cute but small and pretty much unusable as a c...,[],B00F463XV8,B00F9386Q8,AGKASBHYZPGTEPO6LWZPVJWB2BVA,2016-01-13 02:08:01,0,True
3,5,Works great perfect size!,I have lots of different disposable diaper bag...,[],B0007V644S,B07RRDX26B,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,2014-08-25 19:14:11,0,True
4,5,Cute and Works Great,I was so excited for bath time when I register...,[],B002LARFLY,B00OLRJET6,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,2012-10-09 21:42:41,0,False


In [21]:
#đọc ra parquet
cols = ["rating", "parent_asin", "user_id", "timestamp"]
baby_small = baby_product[cols].copy()
#write
out_parquet = r"D:\Study\Amazon\baby_product_small.parquet"
baby_small.to_parquet(out_parquet, index=False)

In [32]:
#import lại
import pandas as pd

path = r"D:\Study\Amazon\baby_product_small.parquet"
baby_product = pd.read_parquet(path)

baby_product.head()

,rating,parent_asin,user_id,timestamp
0,4,B089MS68G8,AGKASBHYZPGTEPO6LWZPVJWB2BVA,2016-08-18 18:52:17
1,5,B01E5E703G,AGKASBHYZPGTEPO6LWZPVJWB2BVA,2016-08-18 17:44:04
2,1,B00F9386Q8,AGKASBHYZPGTEPO6LWZPVJWB2BVA,2016-01-13 02:08:01
3,5,B07RRDX26B,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,2014-08-25 19:14:11
4,5,B00OLRJET6,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,2012-10-09 21:42:41


In [33]:
# dim_product
path = r"D:\Study\Amazon\Baby Products (meta)\meta_Baby_Products.jsonl"
baby_meta = pd.read_json(path, lines=True) 
baby_meta.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Baby,"Chicco Viaro Travel System, Teak",4.6,125,"[Aluminum, Imported, Convenient one-hand quick...","[Product Description, For ultimate convenience...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Viaro Demo Video', 'url': 'https:/...",Chicco,"[Baby Products, Strollers & Accessories, Strol...",{'Product Dimensions': '38 x 41.25 x 25.5 inch...,B01C4319LO,NaN,NaN,NaN
1,AMAZON FASHION,Kisbaby Four Layers Muslin Lightweight Unisex ...,5.0,2,"[95% Cotton, 4 Layer Muslin, Hand Wash in Cold...",[You can choose bigger size If you confuse abo...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Kisbaby,"[Baby Products, Nursery, Bedding, Blankets & S...","{'Material': 'Muslin', 'Color': 'Blue Star', '...",B07FM4MJJP,NaN,NaN,NaN
2,Baby,EZTOTZ Meals with Milton - USA Made Toddler & ...,4.4,37,[WHAT IS MILTON?: Milton is the fun way for yo...,[],22.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],EZTOTZ,[],{'Package Dimensions': '6.3 x 5 x 4.76 inches'...,B08WCG372G,NaN,NaN,NaN
3,Baby,Nuby iMonster Toddler Bowl,4.4,52,[Makes feeding fun for baby and easier for par...,[When babies begin to show interest in feeding...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Nuby,"[Baby Products, Feeding, Solid Feeding, Dishes]",{'Product Dimensions': '2.75 x 5 x 7.5 inches'...,B0083SXABC,NaN,NaN,NaN
4,Amazon Home,mDesign Slim Storage Organizer Container Bin w...,4.5,235,[SMART STORAGE: This large capacity slim bin i...,[The mDesign clear storage bins for kids suppl...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],mDesign,"[Baby Products, Nursery, Furniture, Storage & ...","{'Product Dimensions': '14.62 x 4 x 4 inches',...",B07N8GRHHK,NaN,NaN,NaN


## Elliot

### TF-IDF (Content-based)

#### Step 1: Lấy năm 2022

In [34]:

# convert timestamp
baby_product["timestamp"] = pd.to_datetime(baby_product["timestamp"])

# filter year 2022
# df_2022 = baby_product[baby_product["timestamp"].dt.year == 2022].copy()

#### Bước 2: Map ID về số nguyên (Elliot yêu cầu int)

In [36]:
#cách 1: chạy normally, không cần giảm size
# map user_id -> int
user2uid = {u: i for i, u in enumerate(baby_product["user_id"].unique())}
baby_product["user_int"] = baby_product["user_id"].map(user2uid)

# map parent_asin -> int
asin2iid = {a: i for i, a in enumerate(baby_product["parent_asin"].unique())}
baby_product["item_int"] = baby_product["parent_asin"].map(asin2iid)

In [5]:
### Cách 2: giảm size để test
# giữ top 20k items (bạn có thể thử 5k/10k trước)
TOP_ITEMS = 5000

top_items = (
    df_2022.groupby("item_int").size()
    .sort_values(ascending=False)
    .head(TOP_ITEMS)
    .index
)

df_small = df_2022[df_2022["item_int"].isin(top_items)].copy()

# map user_id -> int
user2uid = {u: i for i, u in enumerate(df_small["user_id"].unique())}
df_small["user_int"] = df_small["user_id"].map(user2uid)

# map parent_asin -> int
asin2iid = {a: i for i, a in enumerate(df_small["parent_asin"].unique())}
df_small["item_int"] = df_small["parent_asin"].map(asin2iid)

#### Bước 3: Tạo dataset.tsv (interaction file)

In [38]:

df_inter = baby_product[["user_int", "item_int", "rating", "timestamp"]].copy()

# timestamp -> unix
df_inter["timestamp"] = df_inter["timestamp"].astype("int64") // 10**9

df_inter.columns = ["userId", "itemId", "rating", "timestamp"]

df_inter.to_csv("dataset.tsv", sep="\t", header=False, index=False)
# dataset_tsv.to_csv(dataset_path, sep="\t",  index=False)

#### Bước  4: Chuẩn hoá baby_meta và map sang item_int

In [39]:
#RÚT GỌN baby_meta (dim_product)

#lấy unique items trong interaction để join 
items_2022 = set(baby_product['parent_asin'].astype(str).unique())

# lấy trường cần thiết
meta = baby_meta[['parent_asin', 'title', 'description']].copy()
meta = meta.dropna(subset=['parent_asin'])

meta['parent_asin'] = meta['parent_asin'].astype(str)

# chỉ giữ item xuất hiện trong interaction 2022
meta = meta[meta['parent_asin'].isin(items_2022)]

In [40]:
#chuẩn hóa description về cùng 1 kiểu string, bỏ Nan,...

meta['title'] = meta['title'].fillna('').astype(str)

def normalize_desc(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ''
    if isinstance(x, list):
        return ' '.join([str(i) for i in x if i is not None])
    if isinstance(x, dict):
        return ' '.join([f"{k} {v}" for k, v in x.items()])
    return str(x)

meta['description'] = meta['description'].apply(normalize_desc)

# map theo item2id 
meta['item_int'] = meta['parent_asin'].map(asin2iid)

meta['item_int'] = meta['item_int'].astype(int)

meta['doc'] = (meta['title'] + ' ' + meta['description']).str.strip()

In [41]:
#lọc data chỉ còn item_int + cột content, nếu xuất hiện 2 item_int thì sẽ group
item_docs = (
    meta.groupby('item_int')['doc']
        .apply(lambda s: ' '.join(s.tolist()))
        .reset_index()
)

#### Bước 5: Tokendize doc (từ text --> chuỗi ký tự)

##### Cách 1: tự biến đổi

In [19]:
import re
from collections import Counter

def tokenize(s: str):
    s = s.lower()
    return re.findall(r"[a-z0-9]+", s)

item_docs['tokens'] = item_docs['doc'].apply(tokenize)

##### Cách 2: dùng sklearn

In [42]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    lowercase=True,
    token_pattern=r"(?u)[a-z0-9]+",  
    min_df=5,
    max_features=50000,
    ngram_range=(1,1),              
)

X = vectorizer.fit_transform(item_docs["doc"].fillna(""))

# vocab: token -> column_index (0..V-1)
vocab = vectorizer.vocabulary_
# list token theo id:
id2token = {i: t for t, i in vocab.items()}

In [43]:
#xuất file
import numpy as np

item_ids = item_docs["item_int"].to_numpy()

# lấy indices (feature ids) cho từng item
with open("item_attributes.tsv", "w", encoding="utf-8") as f:
    for row_idx, item_id in enumerate(item_ids):
        feat_ids = X[row_idx].indices  # các cột != 0
        if feat_ids.size == 0:
            continue

        feat_ids_1based = feat_ids + 1
        f.write("\t".join([str(int(item_id))] + [str(int(x)) for x in feat_ids_1based]) + "\n")

#### Bước 6: Build vocab feature

In [12]:
#lấy hết tất cả vocab trrong cột tokens
all_tokens = []
for toks in item_docs['tokens']:
    all_tokens.extend([t for t in toks if len(t) >= 2])

#đếm tần suất, mục đích chỉ để loại vocab có tần suất xh quqas ít/quá nhiều (b ằng việc set limitr 5 - 50000)
freq = Counter(all_tokens)

min_df = 5
max_features = 50000

#số hóa cho word, tạo 1 vocab dictionary để map sau này
vocab_tokens = [t for t, c in freq.most_common() if c >= min_df][:max_features]
feat2id = {t: i for i, t in enumerate(vocab_tokens, start=1)}

#### Bước 7: Xuất file

* feat2id ở bước 6 là dict: token -> feature_id.

* Kết quả: mỗi item có list số, ví dụ [101, 55, 209, ...].

In [13]:
# viết function mapping từ word --> id (lấy id từ feat2id)
# def tokens_to_feature_ids(tokens):
#     ids = []
#     seen = set()
#     for t in tokens:
#         fid = feat2id.get(t)
#         if fid is not None and fid not in seen:
#             ids.append(fid)
#             seen.add(fid)
#     return ids

# item_docs['feat_ids'] = item_docs['tokens'].apply(tokens_to_feature_ids)

MAX_FEATS_PER_ITEM = 100  

def tokens_to_feature_ids(tokens):
    ids = []
    seen = set()
    for t in tokens:
        fid = feat2id.get(t)
        if fid is not None and fid not in seen:
            ids.append(fid)
            seen.add(fid)
            if len(ids) >= MAX_FEATS_PER_ITEM:
                break
    return ids

item_docs['feat_ids'] = item_docs['tokens'].apply(tokens_to_feature_ids)

item_feat = item_docs[item_docs['feat_ids'].map(len) > 0].copy()

with open('item_features.tsv', 'w', encoding='utf-8') as f:
    for item_id, feat_ids in zip(item_feat['item_int'], item_feat['feat_ids']):
        f.write('\t'.join([str(item_id)] + [str(x) for x in feat_ids]) + '\n')

#### Bước 8: Tạo file yaml

In [44]:
yaml_text = """
experiment:
  dataset: amazon_baby_2022

  data_config:
    strategy: dataset
    dataset_path: C:/Users/admin/Documents/Study/Master/Dissertation/Dataset/dataset.tsv

    side_information:
      - dataloader: ItemAttributes
        attribute_file: C:/Users/admin/Documents/Study/Master/Dissertation/item_attributes.tsv

  splitting:
    save_on_disk: True
    save_folder: C:/Users/admin/Documents/Study/Master/Dissertation/Dataset/splitting/

    test_splitting:
      strategy: temporal_hold_out
      test_ratio: 0.2

  top_k: 50

  evaluation:
    cutoffs: [10, 20, 50]
    simple_metrics: [nDCG, Recall, HR, Precision, MAP, MRR]

  gpu: -1

  models:
    VSM:
      meta:
        save_recs: True
        verbose: True
      similarity: cosine
      user_profile: tfidf
      item_profile: tfidf

"""
cfg_path = r"C:\Users\admin\Documents\Study\Master\Dissertation\elliot\config_files\amazon_vsm_tfidf.yml"

with open(cfg_path, "w", encoding="utf-8") as f:
    f.write(yaml_text)

## SVD

In [ ]:
yaml_text = """
experiment:
  dataset: amazon_baby

  data_config:
    strategy: dataset
    dataset_path: C:/Users/admin/Documents/Study/Master/Dissertation/Dataset/dataset.tsv

  splitting:
    test_splitting:
      strategy: temporal_hold_out
      test_ratio: 0.2

  top_k: 10

  evaluation:
    cutoffs: [10]
    simple_metrics: [nDCG, Recall, HR, Precision, MAP, MRR]

  gpu: -1

  models:
    FunkSVD:
      meta:
        save_recs: True
      epochs: 50
      batch_size: 512
      factors: 64
      lr: 0.001
      reg_w: 0.1
      reg_b: 0.001

"""
cfg_path = r"C:\Users\admin\Documents\Study\Master\Dissertation\elliot\config_files\amazon_svd.yml"

with open(cfg_path, "w", encoding="utf-8") as f:
    f.write(yaml_text)

    # chỗ này thực chất phải là  strategy: temporal_hold_out để cut timestamp

## PureSVD

In [ ]:
yaml_text = """
experiment:
  dataset: amazon_baby

  data_config:
    strategy: dataset
    dataset_path: C:/Users/admin/Documents/Study/Master/Dissertation/Dataset/dataset.tsv

  splitting:
    test_splitting:
      strategy: temporal_hold_out
      folds: 1
      test_ratio: 0.2
      seed: 42

    validation_splitting:
      strategy: random_subsampling
      folds: 1
      validation_ratio: 0.1
      seed: 42

  top_k: 10

  evaluation:
    cutoffs: [10]
    simple_metrics: [nDCG, Recall, HR, Precision, MAP, MRR]

  gpu: -1

  models:
    PureSVD:
      meta:
        save_recs: True
      factors: 200

"""
cfg_path = r"C:\Users\admin\Documents\Study\Master\Dissertation\elliot\config_files\amazon_puresvd.yml"

with open(cfg_path, "w", encoding="utf-8") as f:
    f.write(yaml_text)

    # chỗ này thực chất phải là  strategy: temporal_hold_out để cut timestamp